# 021 指标拆分与可信评价

## 目标
用手算与程序回答：一个数字怎样成为与任务目标相关的证据？主实验采用固定分数 sensor/10，验证成本选择阈值 0.4，冻结后测试平均成本 0.625、F1=4/7。结果只针对原创合成小表。先修 005、018、020；不需要梯度或训练框架。

## 设置
从本单元目录启动。核心计算只用标准库；Jupyter 执行需要 ipykernel。重启内核后顺序运行。以下不提前打开 sealed_test.csv。

In [1]:
from pathlib import Path
import math
from fractions import Fraction
import experiment as ex
UNIT = Path(ex.__file__).resolve().parent
DATA = UNIT / "data"
print("固定正类=1；zero_division=0；分数等于阈值时判正")

固定正类=1；zero_division=0；分数等于阈值时判正


## 步骤 1 验证集手算
读取开发数据。阈值 0.5 应得 TP=2、FP=2、FN=1、TN=3，F1=4/7。

In [2]:
train = ex.load_rows(DATA / "train.csv")
validation = ex.load_rows(DATA / "validation.csv")
y = [r["y"] for r in validation]
scores = ex.score_rows(validation)
m = ex.binary_metrics(y, ex.classify(scores, .5))
assert (m["tp"], m["fp"], m["fn"], m["tn"]) == (2, 2, 1, 3)
assert m["f1"] == float(Fraction(4, 7))
print(m)

{'tn': 3, 'fp': 2, 'fn': 1, 'tp': 2, 'n': 8, 'support': 3, 'predicted_positive': 4, 'accuracy': 0.625, 'precision': 0.5, 'recall': 0.6666666666666666, 'f1': 0.5714285714285714, 'undefined': [], 'zero_division': 0}


### 没有正例不等于检验了正例识别能力
注意 undefined 与数值填充分开。

In [3]:
empty_positive = ex.binary_metrics([0, 0], [0, 0])
assert empty_positive["undefined"] == ["precision", "recall", "f1"]
print(empty_positive)

{'tn': 2, 'fp': 0, 'fn': 0, 'tp': 0, 'n': 2, 'support': 0, 'predicted_positive': 0, 'accuracy': 1.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'undefined': ['precision', 'recall', 'f1'], 'zero_division': 0}


## 步骤 2 只用验证数据比较阈值
成本 = 3×FN+FP；并列选更高阈值。此格只建立候选方案，不读测试数据。

In [4]:
plan, table = ex.make_plan(train, validation, DATA)
for row in table:
    print(row["threshold"], "TP/FP/FN/TN", tuple(row[k] for k in ("tp", "fp", "fn", "tn")), "平均成本", row["mean_cost"])
assert plan["threshold"] == .4
print("冻结阈值:", plan["threshold"])

0.3 TP/FP/FN/TN (3, 3, 0, 2) 平均成本 0.375
0.4 TP/FP/FN/TN (3, 2, 0, 3) 平均成本 0.25
0.5 TP/FP/FN/TN (2, 2, 1, 3) 平均成本 0.625
0.7 TP/FP/FN/TN (2, 1, 1, 4) 平均成本 0.5
0.9 TP/FP/FN/TN (1, 0, 2, 5) 平均成本 0.75
冻结阈值: 0.4


## 步骤 3 执行完整冻结流程
run 先写 frozen_plan.json，再首次读测试，随后检查实体与时间隔离。此函数会重新读取开发数据，输出保存到独立 notebook_outputs。已看过本页输出后，不能继续把该测试集当新证据来挑方案。

In [5]:
result = ex.run(UNIT / "notebook_outputs")
print(result["audit"])
print("测试结果:", result["test"])
assert result["test"]["mean_cost"] == 5/8
assert result["partitions"]["entity_disjoint"] and result["partitions"]["strict_time_order"]

['read_train', 'read_validation', 'select_threshold_using_validation', 'write_frozen_plan', 'read_test_after_freeze']
测试结果: {'tn': 3, 'fp': 2, 'fn': 1, 'tp': 2, 'n': 8, 'support': 3, 'predicted_positive': 4, 'accuracy': 0.625, 'precision': 0.5, 'recall': 0.6666666666666666, 'f1': 0.5714285714285714, 'undefined': [], 'zero_division': 0, 'mean_cost': 0.625}


### 每个分组保留分母
A 的 recall=2/2，B 的 recall=0/1。B 没有正预测，precision 未定义。不要从小组差距直接推断总体差异。

In [6]:
for group, values in result["test_groups"].items():
    print(group, {k: values[k] for k in ("n", "support", "tp", "fp", "fn", "precision", "recall", "undefined")})

A {'n': 4, 'support': 2, 'tp': 2, 'fp': 2, 'fn': 0, 'precision': 0.5, 'recall': 1.0, 'undefined': []}
B {'n': 4, 'support': 1, 'tp': 0, 'fp': 0, 'fn': 1, 'precision': 0.0, 'recall': 0.0, 'undefined': ['precision']}


## 步骤 4 宏平均和微平均
逐类 F1=(8/9,1/2,1/2)。增加缺失类别会改变宏平均的分母与填充项。

In [7]:
mc = result["multiclass"]
print({k: mc[k] for k in ("macro_f1", "micro_f1", "weighted_f1", "accuracy")})
assert math.isclose(mc["macro_f1"], 17/27)
assert mc["micro_f1"] == 10/13
print("纳入缺失类 3 后:", result["multiclass_with_absent_label"]["macro_f1"])
assert math.isclose(result["multiclass_with_absent_label"]["macro_f1"], 17/36)

{'macro_f1': 0.6296296296296297, 'micro_f1': 0.7692307692307693, 'weighted_f1': 0.7991452991452992, 'accuracy': 0.7692307692307693}
纳入缺失类 3 后: 0.4722222222222222


## 步骤 5 回归的量纲
误差=(1,0,-2,1)，MAE=1、MSE=1.5、R²=0.7。常数目标返回 None，JSON 对应 null。

In [8]:
print(result["regression"])
assert result["regression"]["mae"] == 1
assert result["regression"]["mse"] == 1.5
print("常数目标:", ex.regression_metrics([5, 5], [5, 5]))

{'n': 4, 'bias': 0.0, 'mae': 1.0, 'mse': 1.5, 'rmse': 1.224744871391589, 'r2': 0.7, 'r2_undefined_reason': None}
常数目标: {'n': 2, 'bias': 0.0, 'mae': 0.0, 'mse': 0.0, 'rmse': 0.0, 'r2': None, 'r2_undefined_reason': 'n<2 or constant target'}


## 步骤 6 校准分箱是频率诊断
使用独立小例；前三箱左闭右开，最后一箱含 1。每箱只有 2 条，不能从图形声称已校准。

In [9]:
for row in result["calibration"]["bins"]:
    print(row)
print("ECE:", result["calibration"]["ece"], "Brier:", result["calibration"]["brier"])
assert math.isclose(result["calibration"]["ece"], 9/80)
assert math.isclose(result["calibration"]["brier"], 77/400)

{'left': 0.0, 'right': 0.25, 'right_closed': False, 'n': 2, 'positives': 0, 'mean_probability': 0.1, 'positive_frequency': 0.0, 'absolute_gap': 0.1}
{'left': 0.25, 'right': 0.5, 'right_closed': False, 'n': 2, 'positives': 1, 'mean_probability': 0.35, 'positive_frequency': 0.5, 'absolute_gap': 0.15000000000000002}
{'left': 0.5, 'right': 0.75, 'right_closed': False, 'n': 2, 'positives': 1, 'mean_probability': 0.575, 'positive_frequency': 0.5, 'absolute_gap': 0.07499999999999996}
{'left': 0.75, 'right': 1.0, 'right_closed': True, 'n': 2, 'positives': 2, 'mean_probability': 0.875, 'positive_frequency': 1.0, 'absolute_gap': 0.125}
ECE: 0.11249999999999999 Brier: 0.1925


### 粗分箱可以让偏差抵消
粗箱 ECE 降为 0.025，概率预测完全没变。

In [10]:
cal_y = [0,0,1,0,1,0,1,1]
cal_p = [0,.2,.25,.45,.5,.65,.75,1]
coarse = ex.calibration_bins(cal_y, cal_p, [0,.5,1])
assert math.isclose(coarse["ece"], .025)
assert math.isclose(coarse["brier"], 77/400)
print("粗箱 ECE:", coarse["ece"])
print("真实率 0.1 时四次独立观察全负概率:", .9**4)

粗箱 ECE: 0.02500000000000001
真实率 0.1 时四次独立观察全负概率: 0.6561


## 步骤 7 拆分单位
相同九条 panel 记录演示不同的未见条件。要求两种隔离时，时间拆分应报实体重叠。

In [11]:
panel = ex.load_rows(DATA / "split_panel.csv")
entity_parts = ex.split_by_entity(panel, ["A"], ["B"], ["C"])
time_parts = ex.split_by_time(panel, 1, 2)
print("实体拆分:", ex.check_partitions(entity_parts, True, False))
print("时间拆分:", ex.check_partitions(time_parts, False, True))
try:
    ex.check_partitions(time_parts)
except ValueError as error:
    print("预期拒绝:", str(error))
else:
    raise AssertionError("应当拒绝实体交叉")

实体拆分: {'row_counts': [3, 3, 3], 'entity_counts': [1, 1, 1], 'entity_disjoint': True, 'strict_time_order': False}
时间拆分: {'row_counts': [3, 3, 3], 'entity_counts': [3, 3, 3], 'entity_disjoint': False, 'strict_time_order': True}
预期拒绝: Entities overlap across partitions.


## 检查
逐字节比较 Notebook 与随附脚本输出。此比较只验证两种入口一致；独立算术与测试隔离由 test_experiment.py 另行检查。

In [12]:
for name in ex.OUTPUT_NAMES:
    assert (UNIT / "notebook_outputs" / name).read_bytes() == (UNIT / "outputs" / name).read_bytes(), name
print("4 份结果文件逐字节一致。")

4 份结果文件逐字节一致。


## 下一步
对照 answers.md 完成 14 题及实验 A 至 D。报告任务、正类、分母、冻结规则和限制。Notebook 在新进程的真实 InProcessKernel 顺序执行并保存输出；浏览器 UI、跨进程 socket、全新 Anaconda 安装和跨平台安装未测试。来源与定义核查见 source_checks.md。